#### Import Library

In [ ]:
import pandas as pd
from datetime import datetime
import csv

#### Entity / Data CLass

In [2]:
class PatientRecord:
  def __init__(
    self, 
    name: str, 
    age: int, 
    gender: str, 
    blood_type: str, 
    medical_condition: str, 
    date_of_admission: datetime, 
    insurance_provider: str, 
    billing_amount: float, 
    room_number: int, 
    admission_type: str, 
    discharge_date: datetime, 
    medication: str, 
    test_results: str
  ):
    self.name = name
    self.age = age
    self.gender = gender
    self.blood_type = blood_type
    self.medical_condition = medical_condition
    self.date_of_admission = date_of_admission
    self.insurance_provider = insurance_provider
    self.billing_amount = billing_amount
    self.room_number = room_number
    self.admission_type = admission_type
    self.discharge_date = discharge_date
    self.medication = medication
    self.test_results = test_results

  def get_length_of_stay(self) -> int:
    """Menghitung berapa lama pasien dirawat (dalam hari)"""
    return (self.discharge_date - self.date_of_admission).days

  def is_senior_citizen(self) -> bool:
    """Mengecek apakah pasien tergolong lansia (>= 60 tahun)"""
    return self.age >= 60

  def is_high_billing(self, threshold: float = 30000.0) -> bool:
    """Mengecek apakah biaya tagihan pasien termasuk mahal"""
    return self.billing_amount > threshold

#### Repository

In [3]:
class PatientRecordRepository:
  def __init__(self, file_path: str):
    self.file_path = file_path
    self._data = [] 

  def load_csv(self):
    print(f"Membaca file dataset dari: {self.file_path}")
    self._data = [] 
    
    file = open(self.file_path, mode='r', encoding='utf-8')
    reader = csv.DictReader(file)
    
    for row in reader:
      record = PatientRecord(
        name=row['Name'],
        age=int(row['Age']),
        gender=row['Gender'],
        blood_type=row['Blood Type'],
        medical_condition=row['Medical Condition'],
        date_of_admission=datetime.strptime(row['Date of Admission'], '%Y-%m-%d'),
        insurance_provider=row['Insurance Provider'],
        billing_amount=float(row['Billing Amount']),
        room_number=int(row['Room Number']),
        admission_type=row['Admission Type'],
        discharge_date=datetime.strptime(row['Discharge Date'], '%Y-%m-%d'),
        medication=row['Medication'],
        test_results=row['Test Results']
      )
      self._data.append(record)
    
    file.close()

  def get_all_patients(self) -> list:
    return self._data

  def get_patients_by_condition(self, condition: str) -> list:
    filtered_patients = []
    for patient in self._data:
      if patient.medical_condition.lower() == condition.lower():
        filtered_patients.append(patient)
    return filtered_patients

  def get_patients_by_admission_type(self, admission_type: str) -> list:
    filtered_patients = []
    for patient in self._data:
      if patient.admission_type.lower() == admission_type.lower():
        filtered_patients.append(patient)
    return filtered_patients

  def calculate_average_billing(self, provider: str) -> float:
    total_billing = 0.0
    count = 0
    
    for patient in self._data:
      if patient.insurance_provider.lower() == provider.lower():
        total_billing += patient.billing_amount
        count += 1
        
    if count == 0:
      return 0.0
      
    return round(total_billing / count, 2)

#### Service & Analyzer

In [ ]:
class BaseAnalyzer:
  """Superclass yang menyediakan fungsionalitas dasar analisis"""
  def __init__(self, records: list):
    self.records = records

  def get_total_patients(self) -> int:
    return len(self.records)


class OperationalAnalyzer(BaseAnalyzer):
  """Subclass khusus untuk analisis manajemen operasional rumah sakit"""
  
  def avg_stay_by_condition(self) -> dict:
    condition_data = {}
    for r in self.records:
      cond = r.medical_condition
      stay = r.get_length_of_stay()
      if cond not in condition_data:
        condition_data[cond] = {'total_days': 0, 'count': 0}
      condition_data[cond]['total_days'] += stay
      condition_data[cond]['count'] += 1
      
    averages = {}
    for cond, data in condition_data.items():
      averages[cond] = round(data['total_days'] / data['count'], 2)
    return dict(sorted(averages.items(), key=lambda item: item[1], reverse=True))

  def admission_type_distribution(self) -> dict:
    distribution = {}
    for r in self.records:
      adm_type = r.admission_type
      distribution[adm_type] = distribution.get(adm_type, 0) + 1
    return distribution


class FinancialAnalyzer(BaseAnalyzer):
  """Subclass khusus untuk analisis ekonomi kesehatan dan asuransi"""
  
  def avg_billing_by_insurance(self) -> dict:
    insurance_data = {}
    for r in self.records:
      provider = r.insurance_provider
      billing = r.billing_amount
      if provider not in insurance_data:
        insurance_data[provider] = {'total_billing': 0, 'count': 0}
      insurance_data[provider]['total_billing'] += billing
      insurance_data[provider]['count'] += 1
      
    averages = {}
    for provider, data in insurance_data.items():
      averages[provider] = round(data['total_billing'] / data['count'], 2)
    return dict(sorted(averages.items(), key=lambda item: item[1], reverse=True))